# конспект новостей с помощью Gemini

кидаем ссылку, парсим из нее новость, сохраняем в файл, далее прогоняем через модель по ключу и получаем конспект и сохраняем его в вфайл

In [ ]:
# Установка зависимостей (выполните эту ячейку один раз)
%pip install requests beautifulsoup4 pandas python-dotenv google-generativeai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from dotenv import load_dotenv
import google.generativeai as genai

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/g3/p6r0svk56d1cb9z795y19k2h0000gn/T/ipykernel_69017/1380050405.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
# ключ из .env файл
load_dotenv()

url = input("Введите ссылку на новость: ")

In [ ]:
# тут парсинг
print(f"Парсинг данных по ссылке: {url}")
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')
title = soup.find('title').text.strip() if soup.find('title') else "Без заголовка"

# берем текст из разных параграфов
paragraphs = soup.find_all('p')
text_content = "\n".join([p.text.strip() for p in paragraphs if p.text.strip()])

df = pd.DataFrame({'url': [url], 'title': [title], 'content': [text_content]})
df.to_csv('input.csv', index=False, encoding='utf-8')
print("✅ Исходные данные успешно сохранены в input.csv")

Парсинг данных по ссылке: https://www.newsvl.ru/vlad/2026/04/23/237981/
✅ Исходные данные успешно сохранены в input.csv


In [5]:
api_key = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash')

prompt = f"""Пожалуйста, сделай качественное и краткий конспект следующей новости.\n
Верни ответ строго в формате JSON с одним полем \"summary\".\n\n
Заголовок: {title}\n
Текст: {text_content}\n"""

print("Генерация саммаризации с помощью Gemini (ожидается JSON)...\n")
response = model.generate_content(
    prompt,
    generation_config={"response_mime_type": "application/json"}
)

import json
try:
    json.loads(response.text)
except json.JSONDecodeError as exc:
    raise ValueError(f"Gemini вернул невалидный JSON: {response.text}") from exc

full_response_json = response.to_dict()
json_output = json.dumps(full_response_json, ensure_ascii=False, indent=2)
print("✅ Успешно получен полный JSON ответ от API.")
print("\n=== ПОЛНЫЙ ОТВЕТ МОДЕЛИ (JSON) ===\n")
print(json_output)


Генерация саммаризации с помощью Gemini (ожидается JSON)...

✅ Успешно получен полный JSON ответ от API.

=== ПОЛНЫЙ ОТВЕТ МОДЕЛИ (JSON) ===

{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "{\n  \"summary\": \"Во Владивостоке на Змеинке планируется масштабная реорганизация территории площадью 55 гектаров. Проект предусматривает строительство 17 новых жилых домов (высотных и среднеэтажных) для 5200 жителей, а также развитие социальной инфраструктуры: детских садов, образовательного центра, больницы/поликлиники и нескольких спортивных объектов, включая крытый комплекс. Будут возведены коммерческие площади (гостиницы, ТЦ, паркинги). Для этого предстоит снести частный сектор, гаражные кооперативы и ряд заброшенных зданий. Важной частью проекта является сохранение и благоустройство сопки Змеиной как рекреационной зоны. Реализация намечена на две очереди до 2040 года. По проекту проводятся общественные обсуждения до 29 апреля.\"\n}"
          

In [6]:
with open('output_results.json', 'w', encoding='utf-8') as f:
    f.write(json_output + '\n')
print("\n✅ JSON-результат успешно сохранен в output_results.json")


✅ JSON-результат успешно сохранен в output_results.json
